# Fish2 Raphe — consolidated analysis workflow

This notebook is the **single entry point** for the current project.

It brings together:

1. **Selected raphe** — curated ~234 neurons.
2. **Raphe-superior soma-in-ROI** — the filtered Raphe-superior cell-body population.

The notebook uses shared helpers from `src/fish2_raphe/` so clustering, UMAP handling, checked-ID overlays, and static camera conventions are consistent across datasets.

> Large SWCs, cached NBLAST matrices, and figures stay local and are excluded from git.


In [ ]:
# ============================================================
# 1. Imports + project configuration
# ============================================================

from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from fish2_raphe import (
    load_config,
    find_first_existing,
    load_distance_matrix,
    load_assignments,
    clean_distance_matrix,
    hierarchical_recluster,
    compute_umap_from_distance,
    plot_umap_clusters,
    camera_for_static_3d_view,
)

CFG = load_config(PROJECT_ROOT / "configs" / "default.yaml")
OUTPUT_DIR = PROJECT_ROOT / CFG["paths"]["output_dir"]
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Output dir:", OUTPUT_DIR)


In [ ]:
# ============================================================
# 2. Local paths to existing analysis outputs
# ============================================================
# Edit only these paths for your machine.

SELECTED_ROOT = Path(CFG["selected_raphe"]["previous_output_root"])
SOMA_IN_ROI_ROOT = Path(CFG["raphe_superior_soma_in_roi"]["previous_output_root"])

# Example absolute overrides:
# SELECTED_ROOT = Path("/Users/marc/.../fish2_raphe_morphology_v14_selected_vs_raphe_superior_outputs")
# SOMA_IN_ROI_ROOT = Path("/Users/marc/.../fish2_raphe_morphology_v20_rapheSuperior_somaInROI_outputs")

print("Selected root:", SELECTED_ROOT)
print("Soma-in-ROI root:", SOMA_IN_ROI_ROOT)


In [ ]:
# ============================================================
# 3. Robust file discovery for cached assignments/distances
# ============================================================

def discover_dataset_files(root: Path, dataset_key: str):
    root = Path(root)

    assignment_candidates = [
        root / dataset_key / "cluster_assignments.csv",
        root / dataset_key / "assignments.csv",
        root / f"{dataset_key}_cluster_assignments.csv",
        root / f"{dataset_key}_assignments.csv",
        *sorted(root.glob(f"**/*{dataset_key}*assignments*.csv")),
        *sorted(root.glob("**/ACTIVE_*assignments*.csv")),
    ]

    distance_candidates = [
        root / "nblast_cache" / dataset_key / f"{dataset_key}_fishfuncem_distance_rank.pkl",
        root / "nblast_cache" / dataset_key / f"{dataset_key}_fishfuncem_distance_raw_inverse.pkl",
        root / "nblast_cache" / dataset_key / f"{dataset_key}_fishfuncem_distance_rank.csv",
        root / "nblast_cache" / dataset_key / f"{dataset_key}_fishfuncem_distance_raw_inverse.csv",
        *sorted(root.glob(f"**/*{dataset_key}*distance*.pkl")),
        *sorted(root.glob(f"**/*{dataset_key}*distance*.csv")),
        *sorted(root.glob("**/*distance*.pkl")),
        *sorted(root.glob("**/*distance*.csv")),
    ]

    return find_first_existing(assignment_candidates), find_first_existing(distance_candidates)


DATASET_SPECS = {
    "selected_raphe": {"label": "Selected raphe", "root": SELECTED_ROOT},
    "raphe_superior_soma_in_roi": {"label": "Raphe-superior soma-in-ROI", "root": SOMA_IN_ROI_ROOT},
}

datasets = {}

for key, spec in DATASET_SPECS.items():
    try:
        assignment_path, distance_path = discover_dataset_files(spec["root"], key)
        assignments = load_assignments(assignment_path)
        dist = clean_distance_matrix(load_distance_matrix(distance_path), ids=assignments["bodyId"])
        datasets[key] = {
            **spec,
            "assignment_path": assignment_path,
            "distance_path": distance_path,
            "assignments": assignments,
            "distance": dist,
            "embeddings": {},
        }
        print(f"Loaded {key}: {len(assignments):,} assignments; distance {dist.shape}")
        print("  assignments:", assignment_path)
        print("  distance:", distance_path)
    except Exception as e:
        print(f"Could not auto-load {key}: {e}")


## 4. Recluster either dataset without rerunning NBLAST

The cluster cut and UMAP are separate choices:

- **Clustering** uses hierarchical clustering on the cached NBLAST distance matrix.
- **UMAP** is only a 2D visualization of that distance matrix.

So linkage method, cluster number, and UMAP parameters can be explored without recomputing NBLAST.


In [ ]:
# ============================================================
# 4. Reclustering / UMAP configuration
# ============================================================

ACTIVE_DATASET_KEY = "raphe_superior_soma_in_roi"
# ACTIVE_DATASET_KEY = "selected_raphe"

LINKAGE_METHODS_TO_TRY = ["complete", "average", "weighted"]
CLUSTER_K_VALUES = [6, 8, 10, 12, 14, 16, 18, 20]
UMAP_PARAM_GRID = CFG["reclustering"]["umap_grid"]

ACTIVE_LINKAGE_METHOD = "complete"
ACTIVE_CLUSTER_K = 12
ACTIVE_UMAP_NAME = "nn40_md005"

RECLUSTER_DIR = OUTPUT_DIR / "recluster_sweeps" / ACTIVE_DATASET_KEY
RECLUSTER_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# ============================================================
# 5. Run cluster + UMAP variants from cached NBLAST
# ============================================================

ds = datasets[ACTIVE_DATASET_KEY]
dist = ds["distance"]

cluster_variants = {}
for method in LINKAGE_METHODS_TO_TRY:
    for k in CLUSTER_K_VALUES:
        a = hierarchical_recluster(dist, k=k, method=method)
        a["cluster_name"] = a["cluster"].map(
            lambda cl: f"{ACTIVE_DATASET_KEY}_{method}_k{k:02d}_C{int(cl):02d}"
        )
        cluster_variants[(method, int(k))] = a
        a.to_csv(RECLUSTER_DIR / f"assignments_{method}_k{k:02d}.csv", index=False)

umap_variants = {}
for params in UMAP_PARAM_GRID:
    emb = compute_umap_from_distance(
        dist,
        n_neighbors=params["n_neighbors"],
        min_dist=params["min_dist"],
        random_state=42,
    )
    umap_variants[params["name"]] = emb
    emb.to_csv(RECLUSTER_DIR / f"umap_{params['name']}.csv", index=False)

print("Cluster variants:", len(cluster_variants))
print("UMAP variants:", list(umap_variants))


In [ ]:
# ============================================================
# 6. Activate one solution for all downstream steps
# ============================================================

active_assignments = cluster_variants[(ACTIVE_LINKAGE_METHOD, int(ACTIVE_CLUSTER_K))].copy()
active_embedding = umap_variants[ACTIVE_UMAP_NAME].copy()

ds["assignments_original"] = ds["assignments"].copy()
ds["assignments"] = active_assignments
ds["embeddings"] = {"umap": active_embedding}
ds["active_linkage_method"] = ACTIVE_LINKAGE_METHOD
ds["active_k"] = int(ACTIVE_CLUSTER_K)
ds["active_umap_name"] = ACTIVE_UMAP_NAME

print(
    f"Active solution: {ACTIVE_DATASET_KEY} | "
    f"{ACTIVE_LINKAGE_METHOD} | k={ACTIVE_CLUSTER_K} | UMAP={ACTIVE_UMAP_NAME}"
)
display(active_assignments.groupby("cluster").size().rename("n_cells").reset_index())


In [ ]:
# ============================================================
# 7. Figure-ready active UMAP
# ============================================================

fig, ax, active_umap_table = plot_umap_clusters(
    active_embedding,
    active_assignments,
    title=f"{ds['label']} | {ACTIVE_LINKAGE_METHOD} | k={ACTIVE_CLUSTER_K} | {ACTIVE_UMAP_NAME}",
)
display(fig)

umap_out = RECLUSTER_DIR / "ACTIVE_umap.pdf"
fig.savefig(umap_out, bbox_inches="tight")
print("Saved:", umap_out)


## 8. Checked IDs

The same checked-ID list can be used across both datasets. The next cell reports cluster membership and overlays present IDs on the active UMAP.


In [ ]:
# ============================================================
# 8. Checked IDs: membership + UMAP overlay
# ============================================================

CELLS_TO_CHECK = [
    100791363,
    102967931,
    106511646,
    105165440,
    100427087,
    100312010,
    103164598,
    103159060,
    101079563,
    103160069,
    100098958,
    100668438,
    100528051,
    100463138,
    109988161,
]

def check_ids(assignments, ids):
    a = assignments.copy()
    a["bodyId"] = a["bodyId"].astype(int)
    rows = []
    for bid in [int(x) for x in ids]:
        hit = a.loc[a["bodyId"] == bid]
        if len(hit):
            for _, r in hit.iterrows():
                rows.append({
                    "bodyId": bid,
                    "cluster": int(r["cluster"]),
                    "cluster_size": int(r.get("cluster_size", (a["cluster"] == int(r["cluster"])).sum())),
                    "status": "clustered",
                })
        else:
            rows.append({"bodyId": bid, "cluster": np.nan, "cluster_size": np.nan, "status": "not_found"})
    return pd.DataFrame(rows)

checked = check_ids(active_assignments, CELLS_TO_CHECK)
display(checked)
checked.to_csv(RECLUSTER_DIR / "checked_ids_active_solution.csv", index=False)

fig, ax, _ = plot_umap_clusters(
    active_embedding,
    active_assignments,
    highlight_ids=CELLS_TO_CHECK,
    title=f"{ds['label']} — checked IDs on active UMAP",
)
display(fig)
fig.savefig(RECLUSTER_DIR / "checked_ids_on_active_umap.pdf", bbox_inches="tight")


## 9. 3D rendering

For full interactive/static shell rendering, use the dataset-specific notebooks:

- `01_selected_raphe_figures.ipynb`
- `03_raphe_superior_soma_in_roi_recluster_figures.ipynb`

Both use the intended camera convention:

- `top`
- true lateral `side` (left/right, looking along the y-axis)
- `angled`

The helper below is the canonical camera definition for any new code.


In [ ]:
# Canonical static 3D cameras.
STATIC_3D_SIDE_DIRECTION = CFG["static_3d"]["side_direction"]

for view in ["top", "side", "angled", "left", "right", "coronal"]:
    print(view, "->", camera_for_static_3d_view(view, side_direction=STATIC_3D_SIDE_DIRECTION))


## 10. Next step: connectivity and functional integration

The notebook `04_connectivity_next_steps.ipynb` is the scaffold for:

- input/output partner retrieval;
- cluster-level connectivity matrices;
- ROI-resolved connectivity;
- annotations/transmitter metadata;
- integration of functional labels/activity data.
